Sparse PCA - Sparse PCA + GA 

In [ ]:
import time
import warnings
import numpy as np
import pandas as pd

from sklearn.decomposition import SparsePCA
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

from deap import base, creator, tools, algorithms
import random

warnings.filterwarnings("ignore")

# ── 1. Load data ──────────────────────────────────────────────────────────────
X = pd.read_excel('../minmax.xlsx')
y = pd.read_csv('../idC_with_header.csv')['Label']

# ── 2. Train / test split on RAW data FIRST 

X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# ── 3. Sparse PCA  ────────────────────────────────────────────────────────────

N_COMPONENTS = 32
ALPHA        = 1.0   # sparsity regularisation; tune if needed

print(f"Fitting SparsePCA (n_components={N_COMPONENTS}, alpha={ALPHA}) ...")
t0 = time.time()
spca = SparsePCA(n_components=N_COMPONENTS, alpha=ALPHA,
                 random_state=42, n_jobs=-1)

X_train = spca.fit_transform(X_train_raw)   
X_test  = spca.transform(X_test_raw)        

print(f"SparsePCA fit done in {time.time()-t0:.1f}s\n")

# ── 4. Helper: evaluate one classifier and return a result row ────────────────
def evaluate(name, clf, X_tr, X_te, y_tr, y_te, runtime_str):
    clf.fit(X_tr, y_tr)
    preds = clf.predict(X_te)
    return {
        "Classifier": name,
        "Precision" : round(precision_score(y_te, preds, average='weighted'), 2),
        "Recall"    : round(recall_score   (y_te, preds, average='weighted'), 2),
        "F1-Score"  : round(f1_score       (y_te, preds, average='weighted'), 2),
        "Accuracy"  : f"{accuracy_score(y_te, preds)*100:.2f}%",
        "Runtime"   : runtime_str,
    }

# ── 5. SECTION 1 – SparsePCA alone ───────────────────────────────────────────
print("=" * 60)
print("SECTION 1 : SparsePCA (no GA)")
print("=" * 60)

classifiers = {
    "SVM": SVC(random_state=42),
    "RF" : RandomForestClassifier(random_state=42),
    "NB" : GaussianNB(),
}

section1_rows = []
for clf_name, clf in classifiers.items():
    t0  = time.time()
    row = evaluate(clf_name, clf, X_train, X_test, y_train, y_test,
                   runtime_str=f"{time.time()-t0:.1f} sec")
    row["Pipeline"] = "SparsePCA"
    section1_rows.append(row)

df1 = pd.DataFrame(section1_rows)[
    ["Pipeline","Classifier","Precision","Recall","F1-Score","Accuracy","Runtime"]
]
print(df1.to_string(index=False))

# ── 6. GA helper ─────────────────────────────────────────────────────────────
def run_ga(X_tr, y_tr, fitness_clf, n_components):
    """
    Binary GA: each individual is a bit-vector of length n_components.
    A 1 means 'keep this SparsePCA component'.
    Fitness = mean CV accuracy of fitness_clf on the selected components.
    CV is performed on training data only — no leakage.
    """

    # Avoid DEAP re-registration errors when function is called multiple times
    for name in ("FitnessMax", "Individual"):
        if name in creator.__dict__:
            delattr(creator, name)

    creator.create("FitnessMax", base.Fitness, weights=(1.0,))
    creator.create("Individual", list, fitness=creator.FitnessMax)

    tb = base.Toolbox()
    tb.register("attr_bool",  random.randint, 0, 1)
    tb.register("individual", tools.initRepeat, creator.Individual,
                tb.attr_bool, n=n_components)
    tb.register("population", tools.initRepeat, list, tb.individual)

    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    def fitness_fn(individual):
        selected = [i for i, v in enumerate(individual) if v == 1]
        if len(selected) < 2:
            return (0.0,)
        
        scores = cross_val_score(fitness_clf, X_tr[:, selected], y_tr,
                                 cv=skf, scoring='accuracy')
        return (scores.mean(),)

    tb.register("evaluate", fitness_fn)
    tb.register("mate",     tools.cxTwoPoint)
    tb.register("mutate",   tools.mutFlipBit, indpb=0.05)
    tb.register("select",   tools.selTournament, tournsize=3)

    pop   = tb.population(n=100)
    hof   = tools.HallOfFame(1)
    stats = tools.Statistics(lambda ind: ind.fitness.values)
    stats.register("max", np.max)

    algorithms.eaSimple(pop, tb,
                        cxpb=0.7, mutpb=0.2, ngen=50,
                        stats=stats, halloffame=hof, verbose=False)
    best     = hof[0]
    selected = [i for i, v in enumerate(best) if v == 1]
    return selected

# ── 7. SECTION 2 – SparsePCA-GA (fitness matching classifier) ─────────────────
print("\n" + "=" * 60)
print("SECTION 2 : SparsePCA-GA  (fitness matching classifier)")
print("=" * 60)

fitness_classifiers = {
    "SVM": SVC(random_state=42),
    "RF" : RandomForestClassifier(random_state=42),
    "NB" : GaussianNB(),
}

section2_rows = []

for fit_name, fit_clf in fitness_classifiers.items():
    print(f"\n  → GA with {fit_name} fitness function ...")
    t0 = time.time()

    selected_features = run_ga(X_train, y_train, fit_clf, N_COMPONENTS)
    ga_time = time.time() - t0

    rt_str = f"{ga_time:.0f} sec" if ga_time < 60 else f"{ga_time/60:.1f} min"

    clf = fit_clf.__class__(**fit_clf.get_params())

    X_tr_sel = X_train[:, selected_features]
    X_te_sel = X_test [:, selected_features]

    row = evaluate(fit_name, clf, X_tr_sel, X_te_sel, y_train, y_test, rt_str)
    row["Pipeline"] = "SparsePCA-GA"
    section2_rows.append(row)
    print(f"     Selected {len(selected_features)} components | "
          f"Accuracy {row['Accuracy']} | Runtime {rt_str}")

df2 = pd.DataFrame(section2_rows)[
    ["Pipeline","Classifier","Precision","Recall","F1-Score","Accuracy","Runtime"]
]
print("\n")
print(df2.to_string(index=False))

# ── 8. Combined table ─────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("FULL TABLE")
print("=" * 60)
df_full = pd.concat([df1, df2], ignore_index=True)
print(df_full.to_string(index=False))

Tuning: Pipeline: SparsePCA (n=32) → SVM classifier
alpha_values = [0.1, 0.5, 1.0, 2.0, 5.0, 10.0]

In [ ]:
"""c
SparsePCA Parameter Tuning: alpha only
=======================================
Pipeline: SparsePCA (n=32) → SVM classifier
Baseline to beat: 94.38% accuracy (SparsePCA alone, default alpha=1.0)
"""

import time
import warnings
import numpy as np
import pandas as pd

from sklearn.decomposition import SparsePCA
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

warnings.filterwarnings("ignore")
np.random.seed(42)

# ─────────────────────────────────────────────
# 1. LOAD DATA
# ─────────────────────────────────────────────
print("Loading data...")
X = pd.read_excel('../minmax.xlsx').values.astype(np.float32)
y = pd.read_csv('../idC_with_header.csv')['Label'].values

print(f"Dataset: {X.shape[0]} samples x {X.shape[1]} features")
print(f"Baseline to beat: 94.38% (SparsePCA alone, alpha=1.0)\n")

# Fixed train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# ─────────────────────────────────────────────
# 2. ALPHA TUNING LOOP
# ─────────────────────────────────────────────
N_COMPONENTS = 32
alpha_values = [0.1, 0.5, 1.0, 2.0, 5.0, 10.0]

print(f"Tuning alpha over: {alpha_values}")
print(f"(alpha=1.0 is the default)\n")
print(f"{'alpha':>8} {'Accuracy':>10} {'Precision':>10} {'Recall':>10} {'F1':>8} {'Runtime':>10}")
print("-" * 58)

results = []

for alpha in alpha_values:
    t0 = time.time()

    # Step 1: SparsePCA transform (all 32 components used)
    spca = SparsePCA(n_components=N_COMPONENTS, alpha=alpha,
                     random_state=42, n_jobs=-1)
    X_train_latent = spca.fit_transform(X_train)
    X_test_latent  = spca.transform(X_test)

    # Step 2: SVM classifier on all components
    clf = SVC(random_state=42)
    clf.fit(X_train_latent, y_train)
    y_pred = clf.predict(X_test_latent)

    acc  = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, average='weighted', zero_division=0)
    rec  = recall_score(y_test, y_pred, average='weighted', zero_division=0)
    f1   = f1_score(y_test, y_pred, average='weighted', zero_division=0)

    elapsed = time.time() - t0
    rt_str  = f"{elapsed:.0f} sec" if elapsed < 60 else f"{elapsed/60:.1f} min"
    beat    = " ✓" if acc > 0.9438 else ""

    print(f"{alpha:>8} {acc*100:>9.2f}% {prec:>10.2f} {rec:>10.2f} {f1:>8.2f} {rt_str:>10}{beat}")

    results.append({
        "alpha"    : alpha,
        "accuracy" : acc,
        "precision": prec,
        "recall"   : rec,
        "f1"       : f1,
        "runtime"  : rt_str,
    })

# ─────────────────────────────────────────────
# 3. SUMMARY
# ─────────────────────────────────────────────
print("\n" + "=" * 58)
df = pd.DataFrame(results).sort_values("accuracy", ascending=False)
best = df.iloc[0]

print(f"\nBEST RESULT:")
print(f"  alpha     = {best['alpha']}")
print(f"  Accuracy  = {best['accuracy']*100:.2f}%")
print(f"  Precision = {best['precision']:.2f}")
print(f"  Recall    = {best['recall']:.2f}")
print(f"  F1-Score  = {best['f1']:.2f}")

if best['accuracy'] > 0.9438:
    print(f"\n✓ Beats SparsePCA-alone baseline of 94.38%")
else:
    print(f"\n✗ Did not beat SparsePCA-alone baseline of 94.38%")

print("\nFull results (sorted by accuracy):")
print(df.to_string(index=False))

Loading data...
Dataset: 443 samples x 900 features
Baseline to beat: 94.38% (SparsePCA alone, alpha=1.0)

Tuning alpha over: [0.1, 0.5, 1.0, 2.0, 5.0, 10.0]
(alpha=1.0 is the default)

   alpha   Accuracy  Precision     Recall       F1    Runtime
----------------------------------------------------------
     0.1     92.13%       0.93       0.92     0.92    5.3 min
     0.5     93.26%       0.94       0.93     0.93    1.9 min
     1.0     93.26%       0.94       0.93     0.93     32 sec
     2.0     92.13%       0.94       0.92     0.92     12 sec
     5.0     92.13%       0.94       0.92     0.92      5 sec
    10.0     10.11%       0.01       0.10     0.02      1 sec


BEST RESULT:
  alpha     = 0.5
  Accuracy  = 93.26%
  Precision = 0.94
  Recall    = 0.93
  F1-Score  = 0.93

✗ Did not beat SparsePCA-alone baseline of 94.38%

Full results (sorted by accuracy):
 alpha  accuracy  precision   recall       f1 runtime
   0.5  0.932584   0.942288 0.932584 0.932259 1.9 min
   1.0  0.93258

For comparison:
SparsePCA+GA (n=32) → GA (SVM fitness) → SVM classifier
alpha_values = [0.1, 0.5, 1.0, 2.0, 5.0, 10.0]

In [ ]:
"""
SparsePCA Parameter Tuning: alpha only
=======================================
Pipeline: SparsePCA (n=32) → GA (SVM fitness) → SVM classifier
Baseline to beat: 94.38% accuracy (SparsePCA alone, default alpha=1.0)

Tunes:
  - alpha: sparsity penalty on the components
           higher = sparser (fewer original features per component)
           lower  = denser  (closer to standard PCA)


"""

import time
import warnings
import numpy as np
import pandas as pd
import random

from sklearn.decomposition import SparsePCA
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

from deap import base, creator, tools, algorithms

warnings.filterwarnings("ignore")
random.seed(42)
np.random.seed(42)

# ─────────────────────────────────────────────
# 1. LOAD DATA
# ─────────────────────────────────────────────
print("Loading data...")
X = pd.read_excel('../minmax.xlsx').values.astype(np.float32)
y = pd.read_csv('../idC_with_header.csv')['Label'].values

print(f"Dataset: {X.shape[0]} samples x {X.shape[1]} features")
print(f"Baseline to beat: 94.38% (SparsePCA alone, alpha=1.0)\n")

# Fixed train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# ─────────────────────────────────────────────
# 2. GA HELPER (SVM fitness)
# ─────────────────────────────────────────────
N_COMPONENTS = 32

def run_ga_svm_fitness(X_tr, y_tr):
    for name in ("FitnessMax", "Individual"):
        if name in creator.__dict__:
            delattr(creator, name)

    creator.create("FitnessMax", base.Fitness, weights=(1.0,))
    creator.create("Individual", list, fitness=creator.FitnessMax)

    tb = base.Toolbox()
    tb.register("attr_bool",  random.randint, 0, 1)
    tb.register("individual", tools.initRepeat, creator.Individual,
                tb.attr_bool, n=N_COMPONENTS)
    tb.register("population", tools.initRepeat, list, tb.individual)

    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    def fitness_fn(individual):
        selected = [i for i, v in enumerate(individual) if v == 1]
        if len(selected) < 2:
            return (0.0,)
        clf = SVC(random_state=42)
        scores = cross_val_score(clf, X_tr[:, selected], y_tr,
                                 cv=skf, scoring='accuracy')
        return (scores.mean(),)

    tb.register("evaluate", fitness_fn)
    tb.register("mate",     tools.cxTwoPoint)
    tb.register("mutate",   tools.mutFlipBit, indpb=0.05)
    tb.register("select",   tools.selTournament, tournsize=3)

    pop = tb.population(n=100)
    hof = tools.HallOfFame(1)
    algorithms.eaSimple(pop, tb, cxpb=0.7, mutpb=0.2, ngen=50,
                        halloffame=hof, verbose=False)

    best     = hof[0]
    selected = [i for i, v in enumerate(best) if v == 1]
    return selected

# ─────────────────────────────────────────────
# 3. ALPHA TUNING LOOP
# ─────────────────────────────────────────────
alpha_values = [0.1, 0.5, 1.0, 2.0, 5.0, 10.0]
# alpha=1.0 is the default — should reproduce ~93.26% with GA

print(f"Tuning alpha over: {alpha_values}")
print(f"(alpha=1.0 is the default — should reproduce ~93.26% with GA)\n")
print(f"{'alpha':>8} {'Accuracy':>10} {'Precision':>10} {'Recall':>10} {'F1':>8} {'N selected':>12} {'Runtime':>10}")
print("-" * 70)

results = []

for alpha in alpha_values:
    t0 = time.time()

    # Step 1: SparsePCA transform
    spca = SparsePCA(n_components=N_COMPONENTS, alpha=alpha,
                     random_state=42, n_jobs=-1)
    X_train_latent = spca.fit_transform(X_train)
    X_test_latent  = spca.transform(X_test)

    # Step 2: GA with SVM fitness
    selected = run_ga_svm_fitness(X_train_latent, y_train)
    if len(selected) == 0:
        selected = list(range(N_COMPONENTS))  # fallback

    X_tr_sel = X_train_latent[:, selected]
    X_te_sel = X_test_latent[:, selected]

    # Step 3: Final SVM classifier
    clf = SVC(random_state=42)
    clf.fit(X_tr_sel, y_train)
    y_pred = clf.predict(X_te_sel)

    acc  = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, average='weighted', zero_division=0)
    rec  = recall_score(y_test, y_pred, average='weighted', zero_division=0)
    f1   = f1_score(y_test, y_pred, average='weighted', zero_division=0)

    elapsed = time.time() - t0
    rt_str  = f"{elapsed:.0f} sec" if elapsed < 60 else f"{elapsed/60:.1f} min"
    beat    = " ✓" if acc > 0.9438 else ""

    print(f"{alpha:>8} {acc*100:>9.2f}% {prec:>10.2f} {rec:>10.2f} {f1:>8.2f} "
          f"{len(selected):>12} {rt_str:>10}{beat}")

    results.append({
        "alpha"     : alpha,
        "accuracy"  : acc,
        "precision" : prec,
        "recall"    : rec,
        "f1"        : f1,
        "n_selected": len(selected),
        "runtime"   : rt_str,
    })

# ─────────────────────────────────────────────
# 4. SUMMARY
# ─────────────────────────────────────────────
print("\n" + "=" * 70)
df = pd.DataFrame(results).sort_values("accuracy", ascending=False)
best = df.iloc[0]

print(f"\nBEST RESULT:")
print(f"  alpha       = {best['alpha']}")
print(f"  Accuracy    = {best['accuracy']*100:.2f}%")
print(f"  Precision   = {best['precision']:.2f}")
print(f"  Recall      = {best['recall']:.2f}")
print(f"  F1-Score    = {best['f1']:.2f}")
print(f"  GA selected = {int(best['n_selected'])} / {N_COMPONENTS} components")

if best['accuracy'] > 0.9438:
    print(f"\n✓ Beats SparsePCA-alone baseline of 94.38%")
else:
    print(f"\n✗ Did not beat SparsePCA-alone baseline of 94.38%")

print("\nFull results (sorted by accuracy):")
print(df.to_string(index=False))


Loading data...
Dataset: 443 samples x 900 features
Baseline to beat: 94.38% (SparsePCA alone, alpha=1.0)

Tuning alpha over: [0.1, 0.5, 1.0, 2.0, 5.0, 10.0]
(alpha=1.0 is the default — should reproduce ~93.26% with GA)

   alpha   Accuracy  Precision     Recall       F1   N selected    Runtime
----------------------------------------------------------------------
     0.1     89.89%       0.92       0.90     0.90           16    9.5 min
     0.5     94.38%       0.95       0.94     0.94           19    6.7 min ✓
     1.0     93.26%       0.94       0.93     0.93           20    4.5 min
     2.0     92.13%       0.94       0.92     0.92           16    4.5 min
     5.0     91.01%       0.92       0.91     0.91           15    3.6 min
    10.0     10.11%       0.01       0.10     0.02           19    2.9 min


BEST RESULT:
  alpha       = 0.5
  Accuracy    = 94.38%
  Precision   = 0.95
  Recall      = 0.94
  F1-Score    = 0.94
  GA selected = 19 / 32 components

✓ Beats SparsePCA-alone 